# Гипотеза 3. Готовность к командировкам повышает зарплатные ожидания

Проверяем гипотезу:

**соискатели, готовые к командировкам, ожидают более высокую зарплату даже после учета опыта, пола, образования, региона и типа графика.**

Подтверждение считаем успешным, если знак коэффициента в регрессии совпадает с гипотезой, 95% доверительный интервал не пересекает 0 и модуль `t_stat` больше 1.96.

Модель:
- зависимая переменная: `log(salary)`;
- контролируем `experience`, `gender`, `education`, `region_primary_group` и ключевые параметры графика работы.

In [1]:
import os
from pathlib import Path

os.environ["MPLCONFIGDIR"] = str(Path(".mplconfig"))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

In [2]:
def load_prepared_df():
    df = pd.read_csv("dataset_cleaned.csv", parse_dates=["date_publish"], low_memory=False)
    numeric_cols = [
        "salary",
        "experience",
        "birthday",
        "relocation",
        "business_trips",
        "retraining_capability",
        "inner_info_fullness_rate",
        "schedule_type_1",
        "schedule_type_2",
        "schedule_type_3",
        "schedule_type_4",
        "schedule_type_5",
        "schedule_type_6",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "region_group" in df.columns:
        df["region_primary_group"] = df["region_group"].astype("string").str.split("|").str[0]

    if "date_publish" in df.columns and "birthday" in df.columns:
        birthday = pd.to_numeric(df["birthday"], errors="coerce")
        df["age"] = df["date_publish"].dt.year - birthday
        df.loc[~df["age"].between(14, 80), "age"] = np.nan

    df["male"] = df["gender"].astype("string").str.lower().eq("мужской").astype(float)
    df["higher_edu"] = df["education_type"].eq("Высшее").astype(float)
    df["capital_region"] = df["region_primary_group"].eq("Столицы").astype(float)
    df["log_salary"] = np.log(df["salary"])
    return df


def robust_ols(X, y):
    columns = list(X.columns)
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, k = X.shape
    xtx_inv = np.linalg.inv(X.T @ X)
    beta = xtx_inv @ (X.T @ y)
    resid = y - X @ beta

    meat = np.zeros((k, k))
    for i in range(n):
        xi = X[i:i+1].T
        meat += (resid[i] ** 2) * (xi @ xi.T)

    cov = (n / max(n - k, 1)) * xtx_inv @ meat @ xtx_inv
    se = np.sqrt(np.diag(cov))
    t_stat = beta / se
    ci_low = beta - 1.96 * se
    ci_high = beta + 1.96 * se

    result = pd.DataFrame(
        {
            "coef": beta,
            "se": se,
            "t_stat": t_stat,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "pct_effect": np.exp(beta) - 1,
            "pct_effect_ci_low": np.exp(ci_low) - 1,
            "pct_effect_ci_high": np.exp(ci_high) - 1,
        },
        index=columns,
    )
    return result


def build_regression_df(df, target_feature):
    cols = [
        "log_salary",
        "experience",
        "male",
        "higher_edu",
        "relocation",
        "business_trips",
        "schedule_type_1",
        "schedule_type_2",
        "schedule_type_5",
        "schedule_type_6",
        "region_primary_group",
    ]
    reg = df[cols].dropna().copy()
    region_dummies = pd.get_dummies(
        reg["region_primary_group"],
        prefix="region",
        drop_first=True,
        dtype=float,
    )
    X = pd.concat(
        [
            pd.Series(1.0, index=reg.index, name="const"),
            reg[
                [
                    "experience",
                    "male",
                    "higher_edu",
                    "relocation",
                    "business_trips",
                    "schedule_type_1",
                    "schedule_type_2",
                    "schedule_type_5",
                    "schedule_type_6",
                ]
            ].astype(float),
            region_dummies,
        ],
        axis=1,
    )
    result = robust_ols(X, reg["log_salary"])
    return reg, X, result, result.loc[target_feature]

In [3]:
df = load_prepared_df()
reg_df, X, ols_result, target = build_regression_df(df, "business_trips")

print("clean_df shape:", df.shape)
print("regression sample shape:", reg_df.shape)
display(ols_result.loc[["experience", "male", "higher_edu", "relocation", "business_trips", "schedule_type_1", "schedule_type_2", "schedule_type_5", "schedule_type_6"]])
print("\nTarget coefficient:")
target

clean_df shape: (43227, 70)
regression sample shape: (30682, 11)


,coef,se,t_stat,ci_low,ci_high,pct_effect,pct_effect_ci_low,pct_effect_ci_high
experience,0.009327,0.000323,28.855455,0.008693,0.009960,0.009371,0.008731,0.010010
male,0.170610,0.005386,31.674386,0.160053,0.181167,0.186028,0.173573,0.198615
higher_edu,0.182223,0.007492,24.321535,0.167538,0.196908,0.199881,0.182390,0.217631
relocation,0.100698,0.036652,2.747381,0.028859,0.172537,0.105943,0.029280,0.188315
business_trips,0.269725,0.024742,10.901607,0.221231,0.318219,0.309604,0.247612,0.374677
schedule_type_1,0.361350,0.045853,7.880662,0.271479,0.451222,0.435266,0.311903,0.570230
schedule_type_2,-0.042326,0.009282,-4.559934,-0.060520,-0.024133,-0.041443,-0.058725,-0.023844
schedule_type_5,0.041690,0.009168,4.547151,0.023720,0.059660,0.042571,0.024004,0.061476
schedule_type_6,-0.025082,0.006865,-3.653775,-0.038536,-0.011627,-0.024770,-0.037803,-0.011560



Target coefficient:


coef                   0.269725
se                     0.024742
t_stat                10.901607
ci_low                 0.221231
ci_high                0.318219
pct_effect             0.309604
pct_effect_ci_low      0.247612
pct_effect_ci_high     0.374677
Name: business_trips, dtype: float64

In [4]:
desc = (
    df.groupby("business_trips")["salary"]
    .agg(count="size", median="median", mean="mean")
    .rename(index={0.0: "no_business_trips", 1.0: "ready_for_business_trips"})
)
display(desc)

,count,median,mean
business_trips,,,
no_business_trips,40358,20000.0,24384.364785
ready_for_business_trips,2496,30000.0,39975.294872


In [5]:
plt.figure(figsize=(8, 4))
plot_df = pd.DataFrame({
    "coef": [target["coef"]],
    "ci_low": [target["ci_low"]],
    "ci_high": [target["ci_high"]],
}, index=["business_trips"])

plt.errorbar(
    x=plot_df["coef"],
    y=plot_df.index,
    xerr=[plot_df["coef"] - plot_df["ci_low"], plot_df["ci_high"] - plot_df["coef"]],
    fmt="o",
    capsize=5,
)
plt.axvline(0, color="red", linestyle="--", linewidth=1)
plt.title("Коэффициент целевого признака в OLS")
plt.xlabel("Оценка коэффициента на log(salary)")
plt.ylabel("")
plt.tight_layout()
plt.show()

/var/folders/5y/j_6skz8d0y9ggl6xsh5j5zz80000gn/T/ipykernel_25842/614598407.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
is_confirmed = (target["coef"] > 0) and (target["ci_low"] > 0) and (abs(target["t_stat"]) > 1.96)

if is_confirmed:
    print("Гипотеза подтверждена.")
else:
    print("Гипотеза не подтверждена.")

summary = pd.Series({
    "coef": float(target["coef"]),
    "t_stat": float(target["t_stat"]),
    "ci_low": float(target["ci_low"]),
    "ci_high": float(target["ci_high"]),
    "pct_effect": float(target["pct_effect"]),
    "pct_effect_ci_low": float(target["pct_effect_ci_low"]),
    "pct_effect_ci_high": float(target["pct_effect_ci_high"]),
})
summary

Гипотеза подтверждена.


coef                   0.269725
t_stat                10.901607
ci_low                 0.221231
ci_high                0.318219
pct_effect             0.309604
pct_effect_ci_low      0.247612
pct_effect_ci_high     0.374677
dtype: float64

## Вывод

Эта гипотеза подходит для регрессионного подтверждения: коэффициент при `business_trips` интерпретируется как сдвиг зарплатных ожиданий при прочих равных.

Если гипотеза подтверждается, это означает, что мобильность в формате командировок связана не только с описательным ростом зарплаты, но и с устойчивым эффектом внутри модели.